# Automated Regional Impact Auditor (ARIA)
## River Flood Shelter Risk Assessment

**Captain's Log**: Starting analysis of Taiwan's flood risk for emergency shelters using WRA river data and Fire Agency shelter locations.

In [ ]:
# Import required libraries
import geopandas as gpd
import pandas as pd
import folium
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from urllib.parse import quote
import json

# Load environment variables
load_dotenv()

# Buffer distances from .env
BUFFER_HIGH = int(os.getenv('BUFFER_HIGH', 500))
BUFFER_MED = int(os.getenv('BUFFER_MED', 1000))
BUFFER_LOW = int(os.getenv('BUFFER_LOW', 2000))

print(f"Buffer distances: High={BUFFER_HIGH}m, Medium={BUFFER_MED}m, Low={BUFFER_LOW}m")

## 2. 資料來源 (Data Sources)

### A. 資料載入與清理 (Data Ingestion & Cleaning)

**Captain's Log**: 讀取水利署河川面 Shapefile → 檢查 CRS（應為 EPSG:3826）

In [ ]:
# Load river data from WRA with robust download method
import requests
import zipfile
import io
import os

rivers_url = 'https://gic.wra.gov.tw/Gis/gic/API/Google/DownLoad.aspx?fname=RIVERPOLY&filetype=SHP'

print("Loading river data from WRA...")

# Method 1: Try direct reading first (may work in some environments)
try:
    rivers = gpd.read_file(rivers_url)
    print(f"✓ Direct read successful: {len(rivers)} polygons")
except Exception as e:
    print(f"✗ Direct read failed: {e}")
    print("Trying alternative download method...")
    
    # Method 2: Download zip and extract (more reliable)
    try:
        print("Downloading zip file...")
        response = requests.get(rivers_url)
        response.raise_for_status()
        
        # Extract zip to memory
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            # Find the shapefile in the zip
            shp_files = [f for f in zip_ref.namelist() if f.endswith('.shp')]
            
            if shp_files:
                # Extract to temporary directory
                temp_dir = 'temp_river_data'
                os.makedirs(temp_dir, exist_ok=True)
                zip_ref.extractall(temp_dir)
                
                # Find the shapefile path
                shp_path = None
                for root, dirs, files in os.walk(temp_dir):
                    for file in files:
                        if file.endswith('.shp'):
                            shp_path = os.path.join(root, file)
                            break
                    if shp_path:
                        break
                
                if shp_path:
                    rivers = gpd.read_file(shp_path)
                    print(f"✓ Zip extraction successful: {len(rivers)} polygons")
                    print(f"Shapefile path: {shp_path}")
                    
                    # Clean up temporary files
                    import shutil
                    shutil.rmtree(temp_dir)
                else:
                    raise Exception("No shapefile found in extracted zip")
            else:
                raise Exception("No shapefile found in zip file")
                
    except Exception as e2:
        print(f"✗ Zip extraction failed: {e2}")
        
        # Method 3: Use existing local data if available
        local_shp_path = 'river_data/riverpoly/riverpoly.shp'
        if os.path.exists(local_shp_path):
            rivers = gpd.read_file(local_shp_path)
            print(f"✓ Using local data: {len(rivers)} polygons")
        else:
            raise Exception("All methods failed. Please download river data manually.")

print(f"River data loaded: {len(rivers)} polygons")
print(f"Original CRS: {rivers.crs}")
print(f"Bounds: {rivers.total_bounds}")

# Ensure we're working in EPSG:3826 (Taiwan datum)
if rivers.crs != 'EPSG:3826':
    rivers = rivers.to_crs('EPSG:3826')
    print(f"Converted to CRS: {rivers.crs}")

**Captain's Log**: 讀取消防署避難所 CSV → 轉為 GeoDataFrame → 轉換至 **EPSG:3826**

In [ ]:
# Load shelter data (you'll need to download this from data.gov.tw)
# Download from: https://data.gov.tw/dataset/73242
# Save as: data/避難收容處所.csv

try:
    shelters_csv = pd.read_csv('data/避難收容處所.csv', encoding='utf-8')
except UnicodeDecodeError:
    shelters_csv = pd.read_csv('data/避難收容處所.csv', encoding='big5')

print(f"Original shelter records: {len(shelters_csv)}")
print(f"Columns: {list(shelters_csv.columns)}")

# Display first few rows to understand the data structure
shelters_csv.head()

In [ ]:
# **資料清理**：過濾座標為 0 或超出台灣範圍的記錄，記錄清理前後的筆數差異

**Captain's Log**: 讀取鄉鎮界 → 轉換至 **EPSG:3826**

In [ ]:
# Load township boundaries
township_url = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/' + quote('鄉(鎮、市、區)界線1140318.zip')
townships = gpd.read_file(township_url)
townships = townships.to_crs('EPSG:3826')

print(f"Township boundaries loaded: {len(townships)} polygons")
print(f"Township CRS: {townships.crs}")
print(f"Available columns: {list(townships.columns)}")

### B. 多級緩衝區分析 (Multi-Buffer Risk Zoning)

**Captain's Log**: **三級河川警戒緩衝區**：從 `.env` 讀取參數：`BUFFER_HIGH=500`、`BUFFER_MED=1000`、`BUFFER_LOW=2000`（單位：公尺）

In [ ]:
# Create multi-level buffer zones
river_buffers_high = rivers.buffer(BUFFER_HIGH)
river_buffers_med = rivers.buffer(BUFFER_MED)
river_buffers_low = rivers.buffer(BUFFER_LOW)

# Convert to GeoDataFrame
buffer_high_gdf = gpd.GeoDataFrame(geometry=river_buffers_high, crs='EPSG:3826')
buffer_med_gdf = gpd.GeoDataFrame(geometry=river_buffers_med, crs='EPSG:3826')
buffer_low_gdf = gpd.GeoDataFrame(geometry=river_buffers_low, crs='EPSG:3826')

# Dissolve to create unified buffer zones
buffer_high_unified = buffer_high_gdf.dissolve()
buffer_med_unified = buffer_med_gdf.dissolve()
buffer_low_unified = buffer_low_gdf.dissolve()

print(f"High risk buffer (500m): {len(buffer_high_unified)} zones")
print(f"Medium risk buffer (1km): {len(buffer_med_unified)} zones")
print(f"Low risk buffer (2km): {len(buffer_low_unified)} zones")

**Captain's Log**: **空間連接 (Spatial Join)**：`gpd.sjoin()` 找出各級緩衝區內的避難所 → 標記每個避難所的風險等級：高 / 中 / 低 / 安全 → 處理一對多問題：若一個避難所同時落在多個緩衝區，取最高風險等級

In [ ]:
# Spatial joins to identify shelters in each buffer zone
shelters_in_high = gpd.sjoin(shelters, buffer_high_unified, how='inner', predicate='intersects')
shelters_in_med = gpd.sjoin(shelters, buffer_med_unified, how='inner', predicate='intersects')
shelters_in_low = gpd.sjoin(shelters, buffer_low_unified, how='inner', predicate='intersects')

print(f"Shelters in high risk zone (500m): {len(shelters_in_high)}")
print(f"Shelters in medium risk zone (1km): {len(shelters_in_med)}")
print(f"Shelters in low risk zone (2km): {len(shelters_in_low)}")

# CORRECTED: Assign risk levels with proper priority logic
# CRITICAL: Process in order of HIGHEST priority to LOWEST priority
# This ensures highest risk level is preserved when shelters are in multiple buffers
shelters['risk_level'] = 'safe'  # Default to safe

# Step 1: Assign HIGH risk first (highest priority)
shelters.loc[shelters.index.isin(shelters_in_high.index), 'risk_level'] = 'high'
# Step 2: Assign MEDIUM risk only to shelters not already high risk
medium_risk_shelters = shelters_in_med.index.difference(shelters_in_high.index)
shelters.loc[shelters.index.isin(medium_risk_shelters), 'risk_level'] = 'medium'
# Step 3: Assign LOW risk only to shelters not already high or medium risk
low_risk_shelters = shelters_in_low.index.difference(shelters_in_high.index).difference(shelters_in_med.index)
shelters.loc[shelters.index.isin(low_risk_shelters), 'risk_level'] = 'low'

# Count shelters by risk level
risk_counts = shelters['risk_level'].value_counts()
print("\nShelters by risk level:")
for level, count in risk_counts.items():
    print(f"  {level}: {count}")

# Verify priority logic: check overlaps and final assignments
print(f"\nOverlap verification and priority testing:")
print(f"High & Medium overlap: {len(set(shelters_in_high.index) & set(shelters_in_med.index))}")
print(f"High & Low overlap: {len(set(shelters_in_high.index) & set(shelters_in_low.index))}")
print(f"Medium & Low overlap: {len(set(shelters_in_med.index) & set(shelters_in_low.index))}")

# Test specific case: shelters in all three buffers should be HIGH
all_three_overlap = set(shelters_in_high.index) & set(shelters_in_med.index) & set(shelters_in_low.index)
print(f"Shelters in all three buffers: {len(all_three_overlap)}")
if len(all_three_overlap) > 0:
    sample_shelter = list(all_three_overlap)[0]
    final_risk = shelters.loc[sample_shelter, 'risk_level']
    print(f"Sample shelter in all three buffers: Final risk = {final_risk} (should be 'high')")
    if final_risk == 'high':
        print("✓ Priority logic working correctly")
    else:
        print("✗ Priority logic ERROR - should be 'high'")

# Summary of priority logic
print(f"\nPriority Logic Summary:")
print(f"High risk (500m): {len(shelters_in_high)} shelters assigned")
print(f"Medium risk (1km, excluding high): {len(medium_risk_shelters)} shelters assigned")
print(f"Low risk (2km, excluding high/medium): {len(low_risk_shelters)} shelters assigned")
print(f"Safe (outside all buffers): {len(shelters[shelters['risk_level'] == 'safe'])} shelters")

### C. 收容量缺口分析 (Capacity Gap Analysis)

**Captain's Log**: 這是跟 Lab 的關鍵差異——不只看「哪些避難所在風險區」，還要回答實際問題

In [ ]:
# **分區統計**：按鄉鎮市區彙總 → 各區高/中/低風險避難所數量 → 各區風險區內的總收容人數 → 各區安全避難所的總收容人數

In [ ]:
# Calculate capacity gap analysis for each township
def calculate_capacity_gap(row):
    """Calculate capacity gap metrics for a township"""
    # Safe capacity
    safe_capacity = row.get('total_capacity_safe', 0)
    
    # Risk capacity (high + medium + low)
    risk_capacity = (row.get('total_capacity_high', 0) + 
                    row.get('total_capacity_medium', 0) + 
                    row.get('total_capacity_low', 0))
    
    # Total capacity
    total_capacity = safe_capacity + risk_capacity
    
    # Risk shelter count
    risk_shelters = (row.get('shelter_count_high', 0) + 
                   row.get('shelter_count_medium', 0) + 
                   row.get('shelter_count_low', 0))
    
    # Capacity gap ratio (risk shelters vs safe capacity)
    capacity_gap_ratio = risk_shelters / max(safe_capacity, 1) if safe_capacity > 0 else risk_shelters
    
    # Risk score (weighted by risk level and capacity)
    risk_score = (row.get('shelter_count_high', 0) * 3 + 
                 row.get('shelter_count_medium', 0) * 2 + 
                 row.get('shelter_count_low', 0) * 1) * capacity_gap_ratio
    
    return pd.Series({
        'safe_capacity': safe_capacity,
        'risk_capacity': risk_capacity,
        'total_capacity': total_capacity,
        'risk_shelters': risk_shelters,
        'capacity_gap_ratio': capacity_gap_ratio,
        'risk_score': risk_score
    })

# Apply capacity gap analysis
capacity_analysis = township_pivot.apply(calculate_capacity_gap, axis=1)

# Combine with township names
final_analysis = pd.concat([township_pivot[['TOWNNAME']], capacity_analysis], axis=1)

# Sort by comprehensive risk score (not just high-risk shelter count)
top_10_risk_areas = final_analysis.sort_values('risk_score', ascending=False).head(10)

print("Top 10 High-Risk Areas (Comprehensive Analysis):")
for idx, row in top_10_risk_areas.iterrows():
    print(f"{row['TOWNNAME']}:")
    print(f"  Risk Score: {row['risk_score']:.1f}")
    print(f"  Risk Shelters: {int(row['risk_shelters'])}")
    print(f"  Safe Capacity: {int(row['safe_capacity'])}")
    print(f"  Capacity Gap Ratio: {row['capacity_gap_ratio']:.2f}")
    print()

# Export detailed analysis
top_10_risk_areas.to_csv('top_10_risk_areas.csv', index=False, encoding='utf-8-sig')
print("Top 10 risk analysis exported to 'top_10_risk_areas.csv'")

# Summary statistics
print(f"\nCapacity Gap Analysis Summary:")
print(f"Total townships analyzed: {len(final_analysis)}")
print(f"Average safe capacity per township: {final_analysis['safe_capacity'].mean():.1f}")
print(f"Average risk shelters per township: {final_analysis['risk_shelters'].mean():.1f}")
print(f"Average capacity gap ratio: {final_analysis['capacity_gap_ratio'].mean():.2f}")

### D. 視覺化 (Visualization)

**Captain's Log**: **互動式風險地圖**（`.explore()` 或 `folium`）：河川面（藍色）→ 三級緩衝區（紅/橙/黃，半透明）→ 避難所依風險等級著色：紅 = 高風險、橙 = 中風險、黃 = 低風險、綠 = 安全 → 鄉鎮界作為背景 → 點擊避難所可顯示名稱、收容人數、風險等級

In [ ]:
# Create interactive map
# Calculate center of the study area
bounds = shelters.total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

# Convert back to WGS84 for folium
shelters_wgs84 = shelters.to_crs('EPSG:4326')
rivers_wgs84 = rivers.to_crs('EPSG:4326')
buffer_high_wgs84 = buffer_high_unified.to_crs('EPSG:4326')
buffer_med_wgs84 = buffer_med_unified.to_crs('EPSG:4326')
buffer_low_wgs84 = buffer_low_unified.to_crs('EPSG:4326')
townships_wgs84 = townships.to_crs('EPSG:4326')

# Create base map
m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

# Add buffer zones
folium.GeoJson(
    buffer_low_wgs84,
    style_function=lambda x: {'fillColor': 'yellow', 'color': 'none', 'fillOpacity': 0.2},
    name='Low Risk (2km)'
).add_to(m)

folium.GeoJson(
    buffer_med_wgs84,
    style_function=lambda x: {'fillColor': 'orange', 'color': 'none', 'fillOpacity': 0.3},
    name='Medium Risk (1km)'
).add_to(m)

folium.GeoJson(
    buffer_high_wgs84,
    style_function=lambda x: {'fillColor': 'red', 'color': 'none', 'fillOpacity': 0.4},
    name='High Risk (500m)'
).add_to(m)

# Add rivers
folium.GeoJson(
    rivers_wgs84,
    style_function=lambda x: {'color': 'blue', 'weight': 2},
    name='Rivers'
).add_to(m)

# Add shelters with risk-based colors
risk_colors = {
    'high': 'red',
    'medium': 'orange', 
    'low': 'yellow',
    'safe': 'green'
}

for idx, shelter in shelters_wgs84.iterrows():
    risk = shelter['risk_level']
    folium.CircleMarker(
        location=[shelter.geometry.y, shelter.geometry.x],
        radius=5,
        popup=f"Risk: {risk}\nName: {shelter.get('name', 'N/A')}",
        color=risk_colors.get(risk, 'gray'),
        fill=True,
        fillColor=risk_colors.get(risk, 'gray')
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save map
m.save('risk_map_interactive.html')
print("Interactive risk map saved as 'risk_map_interactive.html'")
m

In [ ]:
# **靜態統計圖**：長條圖：Top 10 高風險行政區的避難所數量 vs. 收容量 → 另存 `risk_map.png`

### E. 專業規範 (Infrastructure)

**Captain's Log**: **環境變數**：緩衝距離、目標縣市等參數放在 `.env`，用 `python-dotenv` 讀取 → **Markdown Cells**：每個分析步驟之前寫一段說明（Captain's Log）→ **GitHub**：使用 `gh` CLI 建立 Repo，`.env` 加入 `.gitignore` → **README.md**：包含 **AI 診斷日誌** — 記錄你在過程中遇到的問題及如何解決

In [ ]:
# Export shelter risk audit with complete schema compliance
# Identify required columns for the audit
name_col = None
capacity_col = None

# Find name column
for col in shelters.columns:
    if '名稱' in str(col) or 'name' in str(col).lower():
        name_col = col
        break

# Find capacity column
for col in shelters.columns:
    if '容納人數' in str(col) or 'capacity' in str(col).lower():
        capacity_col = col
        break

print(f"Using name column: {name_col}")
print(f"Using capacity column: {capacity_col}")

# Create comprehensive shelter audit with all required fields
shelter_audit = pd.DataFrame()

# Required fields according to assignment specification
shelter_audit['shelter_id'] = range(len(shelters))  # Unique ID
shelter_audit['name'] = shelters[name_col].fillna('Unknown Shelter') if name_col else 'Unknown Shelter'
shelter_audit['risk_level'] = shelters['risk_level']
shelter_audit['capacity'] = shelters[capacity_col].fillna(0) if capacity_col else 0

# Additional useful fields
shelter_audit['longitude'] = shelters.geometry.x
shelter_audit['latitude'] = shelters.geometry.y

# Calculate distance to nearest river for risk shelters
if 'risk_level' in shelters.columns:
    # For risk shelters, calculate approximate distance to river
    risk_shelters = shelters[shelters['risk_level'] != 'safe'].copy()
    if len(risk_shelters) > 0:
        # Simple distance calculation (in meters, since we're in EPSG:3826)
        distances = []
        for idx, shelter in risk_shelters.iterrows():
            # Find nearest river point (simplified)
            min_distance = float('inf')
            for river_idx, river in rivers.iterrows():
                distance = shelter.geometry.distance(river.geometry)
                if distance < min_distance:
                    min_distance = distance
            distances.append(min_distance)
        
        # Add distances to audit
        risk_distances = dict(zip(risk_shelters.index, distances))
        shelter_audit['distance_to_river'] = shelter_audit['shelter_id'].map(
            lambda x: next((v for k, v in risk_distances.items() if k == x), None)
        )
    else:
        shelter_audit['distance_to_river'] = None

# Validate schema compliance
required_fields = ['shelter_id', 'name', 'risk_level', 'capacity']
missing_fields = [field for field in required_fields if field not in shelter_audit.columns]

if missing_fields:
    print(f"WARNING: Missing required fields: {missing_fields}")
else:
    print("✓ All required fields present in shelter audit")

# Display sample of the audit
print(f"\nShelter Risk Audit Sample (first 3 records):")
print(shelter_audit.head(3).to_string(index=False))

# Export to JSON with proper encoding
try:
    shelter_audit.to_json('shelter_risk_audit.json', orient='records', indent=2, force_ascii=False)
    print(f"\n✓ Shelter risk audit exported to 'shelter_risk_audit.json'")
    print(f"✓ Total records: {len(shelter_audit)}")
    print(f"✓ Schema compliance: {len(missing_fields) == 0}")
except Exception as e:
    print(f"✗ Export failed: {e}")

# Summary statistics
print(f"\n=== FINAL SUMMARY ===")
print(f"Total shelters analyzed: {len(shelters)}")
print(f"High risk shelters: {len(shelters[shelters['risk_level'] == 'high'])}")
print(f"Medium risk shelters: {len(shelters[shelters['risk_level'] == 'medium'])}")
print(f"Low risk shelters: {len(shelters[shelters['risk_level'] == 'low'])}")
print(f"Safe shelters: {len(shelters[shelters['risk_level'] == 'safe'])}")

if capacity_col:
    total_capacity = shelters[capacity_col].sum()
    print(f"Total shelter capacity: {total_capacity:,.0f}")
    print(f"Average capacity per shelter: {total_capacity / len(shelters):.1f}")